# 任意 - Travel Ops Toolbox の SDK 操作

本編の [Lab 4](../labs/04-tools-toolbox.md) は Portal で Toolbox、OpenAPI tool、Skills を追加します。この Notebook は SDK の学習・UI が利用できない場合の補助用で、本編の必須手順ではありません。

この Notebook が作成するのは OpenAPI tool と接続です。Skills の新規アップロードは本編の Portal 手順で行ってください。既存 Toolbox を更新する場合、UI で追加した Skills、他の tool、guardrail は保持します。Skill の自動利用を保証する Notebook ではありません。

**使用する kernel:** `Python (Foundry Workshop)`

認証には `az login` のセッションだけを使います。API key や client secret は読み込みません。上から順に 1 cell ずつ実行してください。

## 1. Repository root を確認する

Notebook をどのフォルダーから開いても、`.workshop/context.json` と既存の Python module を見つけられるようにします。

In [ ]:
import sys
from pathlib import Path


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(
        "Repository root が見つかりません。Codespace でこの Notebook を開いてください。"
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository: {REPO_ROOT.name}")

## 2. Workshop context を読み込む

参加者ごとに異なる endpoint や resource 名は、Lab 1 が生成した `.workshop/context.json` から取得します。値を Notebook に貼り付ける必要はありません。

In [ ]:
from scripts.lib.workshop_context import (
    build_credential,
    load_context,
    project_endpoint,
    terraform_output,
    travel_api_base_url,
)

CONTEXT_PATH = REPO_ROOT / ".workshop" / "context.json"
TOOLBOX_NAME = "contoso-travel-toolbox"
TOOL_NAME = "travel_ops_api"
CONNECTION_NAME = "contoso-travel-toolbox-mcp"
AGENT_NAME = "contoso-travel-assistant"

context = load_context(CONTEXT_PATH)
foundry_endpoint = project_endpoint(context)
project_resource_id = terraform_output(context, "foundry_project_id")
travel_api_url = travel_api_base_url(context)

print(f"Foundry project endpoint: {foundry_endpoint}")
print(f"Travel Ops API: {travel_api_url}")

## 3. Foundry client を作成する

`AzureCliCredential` は、Lab 1 で実行した `az login` のキャッシュを使います。

In [ ]:
from azure.ai.projects import AIProjectClient

credential = build_credential("azure-cli")
client = AIProjectClient(endpoint=foundry_endpoint, credential=credential, allow_preview=True)

print("Foundry client を作成しました。")

## 4. Travel Ops API の OpenAPI 定義を取得する

実際にデプロイされた API の `/openapi.json` を取得します。Container App が停止中でも、既存 helper が有限回だけ再試行します。

In [ ]:
from scripts import create_toolbox

openapi_spec = create_toolbox.fetch_openapi_spec(
    travel_api_url,
    create_toolbox.DEFAULT_OPENAPI_PATH,
)
operation_ids = [
    operation["operationId"]
    for path in openapi_spec["paths"].values()
    for operation in path.values()
    if isinstance(operation, dict) and "operationId" in operation
]

print(f"OpenAPI: {openapi_spec['openapi']}")
print("Operations:", ", ".join(operation_ids))

## 5. OpenAPI tool を定義する

Travel Ops API は合成データだけを返す公開 mock API なので、認証方式は `anonymous` です。実運用 API では、この部分を managed identity などへ置き換えます。

In [ ]:
auth = create_toolbox.build_auth_details("anonymous", audience=None)
travel_ops_tool = create_toolbox.build_openapi_tool(
    tool_name=TOOL_NAME,
    spec=openapi_spec,
    auth=auth,
    description="Contoso Travel Ops API for per-diem, estimates, and simulated preapproval.",
)

print(f"Tool: {travel_ops_tool.name}")

## 6. Toolbox を作成または更新する

同じ OpenAPI 定義で再実行した場合は既存 Toolbox を再利用します。定義が変わった場合だけ新しい version を作り、default に設定します。参加者が version 番号を管理する必要はありません。

In [ ]:
result = create_toolbox.ensure_toolbox(
    client,
    endpoint=foundry_endpoint,
    toolbox_name=TOOLBOX_NAME,
    desired_tool=travel_ops_tool,
)

print(f"Action: {result['action']}")
print(f"Toolbox: {result['toolbox_name']}")
toolbox_endpoint = result["endpoints"]["consumer"]
print(f"MCP endpoint: {toolbox_endpoint}")

## 7. 作成結果を確認する

default の Toolbox に `travel_ops_api` が含まれることを SDK で確認します。Skills をまだアップロードしていない場合は、本編の Portal 手順へ戻って追加してください。

In [ ]:
toolbox = client.toolboxes.get(TOOLBOX_NAME)
toolbox_version = client.toolboxes.get_version(TOOLBOX_NAME, toolbox.default_version)
tool_names = [tool.name for tool in toolbox_version.tools]

assert TOOL_NAME in tool_names, f"{TOOL_NAME} が Toolbox にありません: {tool_names}"
print("Toolbox ready:", ", ".join(tool_names))

## 8. Prompt Agent 用の keyless connection を作成する

Toolbox は MCP endpoint として公開されます。Prompt Agent から secret なしで呼び出せるよう、project managed identity を使う connection を作成します。

In [ ]:
connection = create_toolbox.ensure_toolbox_connection(
    credential=credential,
    project_resource_id=project_resource_id,
    connection_name=CONNECTION_NAME,
    toolbox_endpoint=toolbox_endpoint,
)

print(f"Connection ready: {connection['name']}")

## 9. Toolbox を Prompt Agent に接続する

現在の agent に接続済みの Foundry IQ knowledge base は残したまま、Toolbox MCP tool を追加します。同じ構成で再実行した場合は既存の接続を再利用します。

In [ ]:
agent_result = create_toolbox.attach_toolbox_to_agent(
    client,
    agent_name=AGENT_NAME,
    connection_name=CONNECTION_NAME,
    toolbox_endpoint=toolbox_endpoint,
)

print(f"Agent: {agent_result['agent_name']}")
print(f"Action: {agent_result['action']}")
print("Toolbox connection ready.")

client.close()
credential.close()